<a href="https://colab.research.google.com/github/nilnil47/simple-committee-machine/blob/main/simple_commette_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Minimal Teacher-Student Model (Knowledge Distillation) Implementation

Knowledge Distillation involves training a smaller 'Student' model to mimic the behavior of a pre-trained, larger 'Teacher' model.

### Teacher: Hermite Polynomial $He_3(w_* \cdot x)$

In this setup, the teacher is not a neural network but a fixed direction $w_*$. The target output is the third Hermite polynomial $He_3(z) = z^3 - 3z$ applied to the projection of $x$ onto $w_*$.

In [17]:
import sys
!{sys.executable} -m pip install wandb -q

In [21]:
import wandb

# Centralized Hyperparameters
dimension = 10
n_hidden = 32
batch_size = 256
learning_rate = 0.001
num_samples = 100
train_epochs = 10000

# Initialize a new W&B run
wandb.init(
    project="hermite-distillation",
    config={
        "learning_rate": learning_rate,
        "architecture": "CommitteeStudent",
        "dataset": "Standard Normal Static",
        "epochs": train_epochs,
        "num_samples": num_samples,
        "dimension": dimension,
        "n_hidden": n_hidden,
        "batch_size": batch_size,
        "activation": "erf"
    }
)

avg_projection_z,▆▅▆▅▆██▆▅▆██▆█▆▆█▆▅▅▆▆▆▆█▆▆▅▆▅█▁██▅█▅▆█▆
epoch,▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██████
loss,█▇▇▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
weight_norm,▁▁▁▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
avg_projection_z,-0.09203
epoch,8025
loss,1.37307
weight_norm,44.94042


In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import matplotlib.pyplot as plt

# 1. Define the Teacher
w_star = torch.randn(dimension, 1)
w_star = w_star / torch.norm(w_star)

def hermite_teacher(x, w):
    z = torch.matmul(x, w)
    return (z**3 - 3*z).squeeze()

# 2. Define the Committee Student Model
class CommitteeStudent(nn.Module):
    def __init__(self, d: int, n_hidden: int) -> None:
        super().__init__()
        self.readout_scale = 1.0 / math.sqrt(n_hidden)
        self.W = nn.Parameter(torch.empty(n_hidden, d))
        # Initialize and then normalize each hidden unit to unit norm
        nn.init.normal_(self.W, mean=0.0, std=1.0 / math.sqrt(d))
        with torch.no_grad():
            self.W.data /= torch.norm(self.W, dim=1, keepdim=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = torch.erf(x @ self.W.T)
        return self.readout_scale * h.sum(dim=-1)

student_hermite = CommitteeStudent(dimension, n_hidden)
optimizer = optim.Adam(student_hermite.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

In [23]:
# Training Loop with W&B Logging
import torch
from torch.utils.data import DataLoader, TensorDataset

print(f"Starting training with W&B tracking...")

# 1. Re-ensure the static dataset exists using centralized num_samples
x_data_static = torch.randn(num_samples, dimension)
with torch.no_grad():
    y_data_static = hermite_teacher(x_data_static, w_star)
dataset = TensorDataset(x_data_static, y_data_static)

# 2. Setup DataLoader
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(train_epochs):
    epoch_loss = 0.0
    epoch_z_sum = 0.0
    num_points = 0

    for x_batch, y_batch in train_loader:
        with torch.no_grad():
            # Project x onto w_star
            z = torch.matmul(x_batch, w_star).squeeze()
            epoch_z_sum += z.mean().item()

        y_pred = student_hermite(x_batch)
        loss = criterion(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_points += 1

    # Calculate metrics for the epoch
    avg_epoch_loss = epoch_loss / len(train_loader)
    avg_z = epoch_z_sum / num_points
    with torch.no_grad():
        current_norm = torch.norm(student_hermite.W).item()

    # Log metrics to W&B
    wandb.log({
        "epoch": epoch,
        "loss": avg_epoch_loss,
        "weight_norm": current_norm,
        "avg_projection_z": avg_z
    })

    if (epoch + 1) % 1000 == 0:
        print(f"Epoch [{epoch+1}/{train_epochs}] logged to W&B. Loss: {avg_epoch_loss:.4f}")

# wandb.finish() removed from here to allow cell 9b1c380d to update config

Starting training with W&B tracking...
Epoch [1000/10000] logged to W&B. Loss: 0.7410
Epoch [2000/10000] logged to W&B. Loss: 0.4611
Epoch [3000/10000] logged to W&B. Loss: 0.3708
Epoch [4000/10000] logged to W&B. Loss: 0.3247
Epoch [5000/10000] logged to W&B. Loss: 0.2966
Epoch [6000/10000] logged to W&B. Loss: 0.2818
Epoch [7000/10000] logged to W&B. Loss: 0.2733
Epoch [8000/10000] logged to W&B. Loss: 0.2682
Epoch [9000/10000] logged to W&B. Loss: 0.2655
Epoch [10000/10000] logged to W&B. Loss: 0.2640


In [24]:
# 1. Update the existing run configuration with new metadata
# We use wandb.config.update because the run was already started in cell d13da851
wandb.config.update({
    "w_star": w_star.flatten().tolist(),
    "batch_size": batch_size,
    "activation": "erf"
})

# 2. Log the interactive Plotly figure created in the local training cell
# This allows you to view the interactive version in the W&B dashboard
if 'fig' in globals():
    wandb.log({"training_dynamics_plot": fig})

print("Successfully updated configuration and logged the final plot to W&B.")

# 3. Finalize the run
wandb.finish()

Successfully updated configuration and logged the final plot to W&B.


avg_projection_z,▅▅▁▅▁▅▁▁▅▁▅▅██▅▁▅▁▅▅▅▁▅▅▁▁▁▁▅▅▁▅▅▅▅▁▅▅▁▅
epoch,▁▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇████
loss,█▇▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
weight_norm,▁▁▁▁▁▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇██
avg_projection_z,-0.14469
epoch,9999
loss,0.26396
weight_norm,41.37061


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